## ETL

### Acknowledgments

##### Data provided by Open Source Mental Health (OSMH) https://osmhhelp.org/research.html
##### The following ETL script is primarily provided by Mariam Raafat Mohamed in 'Mental Health Tech: 9 Years of Insights & Prediction' https://www.kaggle.com/code/mariamraafatbrownies/mental-health-tech-9-years-of-insights-prediction/notebook
##### I've modified and changed methodology where relevant, including:
> ##### Keeping variables that were dropped for further analysis in EDA
> ##### Different method for encoding responses
#####
##### All code in subsequent notebooks is my own work

### Load relevant libraries

In [1]:
from pathlib import Path
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

import numpy as np

### Read in the Data

In [2]:
# Resolve repository root whether notebook is launched
# from the repository root or notebooks/ directory.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

RAW_DATA = REPO_ROOT / "data" / "raw"
PROCESSED_DATA = REPO_ROOT / "data" / "processed"

In [3]:
df_2014 = pd.read_csv(RAW_DATA / 'survey.csv')
df_2016 = pd.read_csv(RAW_DATA / 'mental-heath-in-tech-2016_20161114.csv')
df_2017 = pd.read_csv(RAW_DATA / 'OSMI Mental Health in Tech Survey 2017.csv')
df_2018 = pd.read_csv(RAW_DATA / 'OSMI Mental Health in Tech Survey 2018.csv')
df_2019 = pd.read_csv(RAW_DATA / 'OSMI 2019 Mental Health in Tech Survey Results - OSMI Mental Health in Tech Survey 2019.csv')
df_2020 = pd.read_csv(RAW_DATA / 'OSMI 2020 Mental Health in Tech Survey Results .csv')
df_2021 = pd.read_csv(RAW_DATA / 'OSMI 2021 Mental Health in Tech Survey Results .csv')
df_2022 = pd.read_csv(RAW_DATA / 'responses_2022.csv')
df_2023 = pd.read_csv(RAW_DATA / 'responses_2023.csv')

<h3 style="color:#1b1d1f">Understanding the Columns:</h3>

| Column Name               | Question                                                                                   |
|--------------------------|--------------------------------------------------------------------------------------------|
| Timestamp                | Timestamp                                                                                 |
| Age                      | Age                                                                                        |
| Gender                   | Gender                                                                                     |
| Country                  | Country                                                                                    |
| state                    | If you live in the United States, which state or territory do you live in?                |
| self_employed            | Are you self-employed?                                                                     |
| family_history           | Do you have a family history of mental illness?                                            |
| treatment                | Have you sought treatment for a mental health condition?                                   |
| work_interfere           | If you have a mental health condition, do you feel that it interferes with your work?      |
| no_employees             | How many employees does your company or organization have?                                 |
| remote_work              | Do you work remotely (outside of an office) at least 50% of the time?                      |
| tech_company             | Is your employer primarily a tech company/organization?                                    |
| benefits                 | Does your employer provide mental health benefits?                                         |
| care_options             | Do you know the options for mental health care your employer provides?                     |
| wellness_program         | Has your employer ever discussed mental health as part of an employee wellness program?    |
| seek_help                | Does your employer provide resources to learn more about mental health issues and how to seek help? |
| anonymity                | Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources? |
| leave                    | How easy is it for you to take medical leave for a mental health condition?                |
| mental_health_consequence| Do you think that discussing a mental health issue with your employer would have negative consequences? |
| phys_health_consequence  | Do you think that discussing a physical health issue with your employer would have negative consequences? |
| coworkers                | Would you be willing to discuss a mental health issue with your coworkers?                 |
| supervisor               | Would you be willing to discuss a mental health issue with your direct supervisor(s)?      |
| mental_health_interview  | Would you bring up a mental health issue with a potential employer in an interview?        |
| phys_health_interview    | Would you bring up a physical health issue with a potential employer in an interview?      |
| mental_vs_physical       | Do you feel that your employer takes mental health as seriously as physical health?        |
| obs_consequence          | Have you heard of or observed negative consequences for coworkers with mental health conditions in your workplace? |
| comments                 | Any additional notes or comments                                                           |


### Summarize data

In [4]:
df_2014.head()

,Timestamp,Age,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,remote_work,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
0,2014-08-27 11:29:31,37,Female,United States,IL,NaN,No,Yes,Often,6-25,No,Yes,Yes,Not sure,No,Yes,Yes,Somewhat easy,No,No,Some of them,Yes,No,Maybe,Yes,No,NaN
1,2014-08-27 11:29:37,44,M,United States,IN,NaN,No,No,Rarely,More than 1000,No,No,Don't know,No,Don't know,Don't know,Don't know,Don't know,Maybe,No,No,No,No,No,Don't know,No,NaN
2,2014-08-27 11:29:44,32,Male,Canada,NaN,NaN,No,No,Rarely,6-25,No,Yes,No,No,No,No,Don't know,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,No,NaN
3,2014-08-27 11:29:46,31,Male,United Kingdom,NaN,NaN,Yes,Yes,Often,26-100,No,Yes,No,Yes,No,No,No,Somewhat difficult,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,NaN
4,2014-08-27 11:30:22,31,Male,United States,TX,NaN,No,No,Never,100-500,Yes,Yes,Yes,No,Don't know,Don't know,Don't know,Don't know,No,No,Some of them,Yes,Yes,Yes,Don't know,No,NaN


In [5]:
df_2016.head()

,Are you self-employed?,How many employees does your company or organization have?,Is your employer primarily a tech company/organization?,Is your primary role within your company related to tech/IT?,Does your employer provide mental health benefits as part of healthcare coverage?,Do you know the options for mental health care available under your employer-provided coverage?,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",Does your employer offer resources to learn more about mental health concerns and options for seeking help?,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,"If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:",Do you think that discussing a mental health disorder with your employer would have negative consequences?,Do you think that discussing a physical health issue with your employer would have negative consequences?,Would you feel comfortable discussing a mental health disorder with your coworkers?,Would you feel comfortable discussing a mental health disorder with your direct supervisor(s)?,Do you feel that your employer takes mental health as seriously as physical health?,Have you heard of or observed negative consequences for co-workers who have been open about mental health issues in your workplace?,Do you have medical coverage (private insurance or state-provided) which includes treatment of mental health issues?,Do you know local or online resources to seek help for a mental health disorder?,"If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?","If you have revealed a mental health issue to a client or business contact, do you believe this has impacted you negatively?","If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?","If you have revealed a mental health issue to a coworker or employee, do you believe this has impacted you negatively?",Do you believe your productivity is ever affected by a mental health issue?,"If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?",Do you have previous employers?,Have your previous employers provided mental health benefits?,Were you aware of the options for mental health care provided by your previous employers?,Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?,Did your previous employers provide resources to learn more about mental health issues and how to seek help?,Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?,Do you think that discussing a mental health disorder with previous employers would have negative consequences?,Do you think that discussing a physical health issue with previous employers would have negative consequences?,Would you have been willing to discuss a mental health issue with your previous co-workers?,Would you have been willing to discuss a mental health issue with your direct supervisor(s)?,Did you feel that your previous employers took mental health as seriously as physical health?,Did you hear of or observe negative consequences for co-workers with mental health issues in your previous workplaces?,Would you be willing to bring up a physical health issue with a potential employer in an interview?,Why or why not?,Would you bring up a mental health issue with a potential employer in an interview?,Why or why not?.1,Do you feel that being identified as a person with a mental health issue would hurt your career?,Do you think that team members/co-workers would view you more negatively if they knew you suffered from a menta

In [6]:
df_2017.head()

,#,<strong>Are you self-employed?</strong>,How many employees does your company or organization have?,Is your employer primarily a tech company/organization?,Is your primary role within your company related to tech/IT?,Does your employer provide mental health benefits as part of healthcare coverage?,Do you know the options for mental health care available under your employer-provided health coverage?,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",Does your employer offer resources to learn more about mental health disorders and options for seeking help?,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,"If a mental health issue prompted you to request a medical leave from work, how easy or difficult would it be to ask for that leave?",Would you feel more comfortable talking to your coworkers about your physical health or your mental health?,Would you feel comfortable discussing a mental health issue with your direct supervisor(s)?,Have you ever discussed your mental health with your employer?,"Describe the conversation you had with your employer about your mental health, including their reactions and what actions were taken to address your mental health issue/questions.",Would you feel comfortable discussing a mental health issue with your coworkers?,Have you ever discussed your mental health with coworkers?,Describe the conversation with coworkers you had about your mental health including their reactions.,Have you ever had a coworker discuss their or another coworker's mental health with you?,Describe the conversation your coworker had with you about their mental health (please do not use names).,"Overall, how much importance does your employer place on physical health?","Overall, how much importance does your employer place on mental health?",Do you have medical coverage (private insurance or state-provided) that includes treatment of mental health disorders?,Do you know local or online resources to seek help for a mental health issue?,"<strong>If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?</strong>","If you have revealed a mental health disorder to a client or business contact, how has this affected you or the relationship?","<strong>If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?</strong>","If you have revealed a mental health disorder to a coworker or employee, how has this impacted you or the relationship?",Do you believe your productivity is ever affected by a mental health issue?,"If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?",<strong>Do you have previous employers?</strong>,Was your employer primarily a tech company/organization?,<strong>Have your previous employers provided mental health benefits?</strong>,<strong>Were you aware of the options for mental health care provided by your previous employers?</strong>,Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?,Did your previous employers provide resources to learn more about mental health disorders and how to seek help?,Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?,Would you have felt more comfortable talking to your previous employer about your physical health or your mental health?,Would you have been willing to discuss your mental health with your direct supervisor(s)?,Did you ever discuss your mental health with your previous employer?,"Describe the conversation you had with your previous employer about your mental health, including their reactions and actions taken to address yo

In [7]:
df_2018.head()

,#,<strong>Are you self-employed?</strong>,How many employees does your company or organization have?,Is your employer primarily a tech company/organization?,Is your primary role within your company related to tech/IT?,Does your employer provide mental health benefits as part of healthcare coverage?,Do you know the options for mental health care available under your employer-provided health coverage?,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",Does your employer offer resources to learn more about mental health disorders and options for seeking help?,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,"If a mental health issue prompted you to request a medical leave from work, how easy or difficult would it be to ask for that leave?",Would you feel more comfortable talking to your coworkers about your physical health or your mental health?,Would you feel comfortable discussing a mental health issue with your direct supervisor(s)?,Have you ever discussed your mental health with your employer?,"Describe the conversation you had with your employer about your mental health, including their reactions and what actions were taken to address your mental health issue/questions.",Would you feel comfortable discussing a mental health issue with your coworkers?,Have you ever discussed your mental health with coworkers?,Describe the conversation with coworkers you had about your mental health including their reactions.,Have you ever had a coworker discuss their or another coworker's mental health with you?,Describe the conversation your coworker had with you about their mental health (please do not use names).,"Overall, how much importance does your employer place on physical health?","Overall, how much importance does your employer place on mental health?",Do you have medical coverage (private insurance or state-provided) that includes treatment of mental health disorders?,Do you know local or online resources to seek help for a mental health issue?,"<strong>If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?</strong>","If you have revealed a mental health disorder to a client or business contact, how has this affected you or the relationship?","<strong>If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?</strong>","If you have revealed a mental health disorder to a coworker or employee, how has this impacted you or the relationship?",Do you believe your productivity is ever affected by a mental health issue?,"If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?",<strong>Do you have previous employers?</strong>,Was your employer primarily a tech company/organization?,<strong>Have your previous employers provided mental health benefits?</strong>,<strong>Were you aware of the options for mental health care provided by your previous employers?</strong>,Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?,Did your previous employers provide resources to learn more about mental health disorders and how to seek help?,Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?,Would you have felt more comfortable talking to your previous employer about your physical health or your mental health?,Would you have been willing to discuss your mental health with your direct supervisor(s)?,Did you ever discuss your mental health with your previous employer?,"Describe the conversation you had with your previous employer about your mental health, including their reactions and actions taken to address yo

In [8]:
df_2019.head()

,*Are you self-employed?*,How many employees does your company or organization have?,Is your employer primarily a tech company/organization?,Is your primary role within your company related to tech/IT?,Does your employer provide mental health benefits as part of healthcare coverage?,Do you know the options for mental health care available under your employer-provided health coverage?,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",Does your employer offer resources to learn more about mental health disorders and options for seeking help?,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,"If a mental health issue prompted you to request a medical leave from work, how easy or difficult would it be to ask for that leave?",Would you feel more comfortable talking to your coworkers about your physical health or your mental health?,Would you feel comfortable discussing a mental health issue with your direct supervisor(s)?,Have you ever discussed your mental health with your employer?,"Describe the conversation you had with your employer about your mental health, including their reactions and what actions were taken to address your mental health issue/questions.",Would you feel comfortable discussing a mental health issue with your coworkers?,Have you ever discussed your mental health with coworkers?,Describe the conversation with coworkers you had about your mental health including their reactions.,Have you ever had a coworker discuss their or another coworker's mental health with you?,Describe the conversation your coworker had with you about their mental health (please do not use names).,"Overall, how much importance does your employer place on physical health?","Overall, how much importance does your employer place on mental health?",Do you have medical coverage (private insurance or state-provided) that includes treatment of mental health disorders?,Do you know local or online resources to seek help for a mental health issue?,"If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?","If you have revealed a mental health disorder to a client or business contact, how has this affected you or the relationship?","If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?","If you have revealed a mental health disorder to a coworker or employee, how has this impacted you or the relationship?",Do you believe your productivity is ever affected by a mental health issue?,"If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?",*Do you have previous employers?*,Was your employer primarily a tech company/organization?,Have your previous employers provided mental health benefits?,Were you aware of the options for mental health care provided by your previous employers?,Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?,Did your previous employers provide resources to learn more about mental health disorders and how to seek help?,Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?,Would you have felt more comfortable talking to your previous employer about your physical health or your mental health?,Would you have been willing to discuss your mental health with your direct supervisor(s)?,Did you ever discuss your mental health with your previous employer?,"Describe the conversation you had with your previous employer about your mental health, including their reactions and actions taken to address your mental health issue/questions.",Would you have been willing to discuss your mental health with yo

In [9]:
df_2020.head()

,#,*Are you self-employed?*,How many employees does your company or organization have?,Is your employer primarily a tech company/organization?,Is your primary role within your company related to tech/IT?,Does your employer provide mental health benefits as part of healthcare coverage?,Do you know the options for mental health care available under your employer-provided health coverage?,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",Does your employer offer resources to learn more about mental health disorders and options for seeking help?,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,"If a mental health issue prompted you to request a medical leave from work, how easy or difficult would it be to ask for that leave?",Would you feel more comfortable talking to your coworkers about your physical health or your mental health?,Would you feel comfortable discussing a mental health issue with your direct supervisor(s)?,Have you ever discussed your mental health with your employer?,"Describe the conversation you had with your employer about your mental health, including their reactions and what actions were taken to address your mental health issue/questions.",Would you feel comfortable discussing a mental health issue with your coworkers?,Have you ever discussed your mental health with coworkers?,Describe the conversation with coworkers you had about your mental health including their reactions.,Have you ever had a coworker discuss their or another coworker's mental health with you?,Describe the conversation your coworker had with you about their mental health (please do not use names).,"Overall, how much importance does your employer place on physical health?","Overall, how much importance does your employer place on mental health?",Do you have medical coverage (private insurance or state-provided) that includes treatment of mental health disorders?,Do you know local or online resources to seek help for a mental health issue?,"If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?","If you have revealed a mental health disorder to a client or business contact, how has this affected you or the relationship?","If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?","If you have revealed a mental health disorder to a coworker or employee, how has this impacted you or the relationship?",Do you believe your productivity is ever affected by a mental health issue?,"If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?",*Do you have previous employers?*,Was your employer primarily a tech company/organization?,Have your previous employers provided mental health benefits?,Were you aware of the options for mental health care provided by your previous employers?,Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?,Did your previous employers provide resources to learn more about mental health disorders and how to seek help?,Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?,Would you have felt more comfortable talking to your previous employer about your physical health or your mental health?,Would you have been willing to discuss your mental health with your direct supervisor(s)?,Did you ever discuss your mental health with your previous employer?,"Describe the conversation you had with your previous employer about your mental health, including their reactions and actions taken to address your mental health issue/questions.",Would you have been willing to discuss your mental health with 

In [10]:
df_2021.head()

,#,*Are you self-employed?*,How many employees does your company or organization have?,Is your employer primarily a tech company/organization?,Is your primary role within your company related to tech/IT?,Does your employer provide mental health benefits as part of healthcare coverage?,Do you know the options for mental health care available under your employer-provided health coverage?,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",Does your employer offer resources to learn more about mental health disorders and options for seeking help?,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,"If a mental health issue prompted you to request a medical leave from work, how easy or difficult would it be to ask for that leave?",Would you feel more comfortable talking to your coworkers about your physical health or your mental health?,Would you feel comfortable discussing a mental health issue with your direct supervisor(s)?,Have you ever discussed your mental health with your employer?,"Describe the conversation you had with your employer about your mental health, including their reactions and what actions were taken to address your mental health issue/questions.",Would you feel comfortable discussing a mental health issue with your coworkers?,Have you ever discussed your mental health with coworkers?,Describe the conversation with coworkers you had about your mental health including their reactions.,Have you ever had a coworker discuss their or another coworker's mental health with you?,Describe the conversation your coworker had with you about their mental health (please do not use names).,"Overall, how much importance does your employer place on physical health?","Overall, how much importance does your employer place on mental health?",Do you have medical coverage (private insurance or state-provided) that includes treatment of mental health disorders?,Do you know local or online resources to seek help for a mental health issue?,"If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?","If you have revealed a mental health disorder to a client or business contact, how has this affected you or the relationship?","If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?","If you have revealed a mental health disorder to a coworker or employee, how has this impacted you or the relationship?",Do you believe your productivity is ever affected by a mental health issue?,"If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?",*Do you have previous employers?*,Was your employer primarily a tech company/organization?,Have your previous employers provided mental health benefits?,Were you aware of the options for mental health care provided by your previous employers?,Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?,Did your previous employers provide resources to learn more about mental health disorders and how to seek help?,Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?,Would you have felt more comfortable talking to your previous employer about your physical health or your mental health?,Would you have been willing to discuss your mental health with your direct supervisor(s)?,Did you ever discuss your mental health with your previous employer?,"Describe the conversation you had with your previous employer about your mental health, including their reactions and actions taken to address your mental health issue/questions.",Would you have been willing to discuss your mental health with 

In [11]:
df_2022.head()

,#,*Are you self-employed?*,How many employees does your company or organization have?,Is your employer primarily a tech company/organization?,Is your primary role within your company related to tech/IT?,Does your employer provide mental health benefits as part of healthcare coverage?,Do you know the options for mental health care available under your employer-provided health coverage?,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",Does your employer offer resources to learn more about mental health disorders and options for seeking help?,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,"If a mental health issue prompted you to request a medical leave from work, how easy or difficult would it be to ask for that leave?",Would you feel more comfortable talking to your coworkers about your physical health or your mental health?,Would you feel comfortable discussing a mental health issue with your direct supervisor(s)?,Have you ever discussed your mental health with your employer?,"Describe the conversation you had with your employer about your mental health, including their reactions and what actions were taken to address your mental health issue/questions.",Would you feel comfortable discussing a mental health issue with your coworkers?,Have you ever discussed your mental health with coworkers?,Describe the conversation with coworkers you had about your mental health including their reactions.,Have you ever had a coworker discuss their or another coworker's mental health with you?,Describe the conversation your coworker had with you about their mental health (please do not use names).,"Overall, how much importance does your employer place on physical health?","Overall, how much importance does your employer place on mental health?",Do you have medical coverage (private insurance or state-provided) that includes treatment of mental health disorders?,Do you know local or online resources to seek help for a mental health issue?,"If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?","If you have revealed a mental health disorder to a client or business contact, how has this affected you or the relationship?","If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?","If you have revealed a mental health disorder to a coworker or employee, how has this impacted you or the relationship?",Do you believe your productivity is ever affected by a mental health issue?,"If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?",*Do you have previous employers?*,Was your employer primarily a tech company/organization?,Have your previous employers provided mental health benefits?,Were you aware of the options for mental health care provided by your previous employers?,Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?,Did your previous employers provide resources to learn more about mental health disorders and how to seek help?,Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?,Would you have felt more comfortable talking to your previous employer about your physical health or your mental health?,Would you have been willing to discuss your mental health with your direct supervisor(s)?,Did you ever discuss your mental health with your previous employer?,"Describe the conversation you had with your previous employer about your mental health, including their reactions and actions taken to address your mental health issue/questions.",Would you have been willing to discuss your mental health with 

In [12]:
df_2023.head()

,#,*Are you self-employed?*,How many employees does your company or organization have?,Is your employer primarily a tech company/organization?,Is your primary role within your company related to tech/IT?,Does your employer provide mental health benefits as part of healthcare coverage?,Do you know the options for mental health care available under your employer-provided health coverage?,"Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",Does your employer offer resources to learn more about mental health disorders and options for seeking help?,Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,"If a mental health issue prompted you to request a medical leave from work, how easy or difficult would it be to ask for that leave?",Would you feel more comfortable talking to your coworkers about your physical health or your mental health?,Would you feel comfortable discussing a mental health issue with your direct supervisor(s)?,Have you ever discussed your mental health with your employer?,"Describe the conversation you had with your employer about your mental health, including their reactions and what actions were taken to address your mental health issue/questions.",Would you feel comfortable discussing a mental health issue with your coworkers?,Have you ever discussed your mental health with coworkers?,Describe the conversation with coworkers you had about your mental health including their reactions.,Have you ever had a coworker discuss their or another coworker's mental health with you?,Describe the conversation your coworker had with you about their mental health (please do not use names).,"Overall, how much importance does your employer place on physical health?","Overall, how much importance does your employer place on mental health?",Do you have medical coverage (private insurance or state-provided) that includes treatment of mental health disorders?,Do you know local or online resources to seek help for a mental health issue?,"If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?","If you have revealed a mental health disorder to a client or business contact, how has this affected you or the relationship?","If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?","If you have revealed a mental health disorder to a coworker or employee, how has this impacted you or the relationship?",Do you believe your productivity is ever affected by a mental health issue?,"If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?",*Do you have previous employers?*,Was your employer primarily a tech company/organization?,Have your previous employers provided mental health benefits?,Were you aware of the options for mental health care provided by your previous employers?,Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?,Did your previous employers provide resources to learn more about mental health disorders and how to seek help?,Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?,Would you have felt more comfortable talking to your previous employer about your physical health or your mental health?,Would you have been willing to discuss your mental health with your direct supervisor(s)?,Did you ever discuss your mental health with your previous employer?,"Describe the conversation you had with your previous employer about your mental health, including their reactions and actions taken to address your mental health issue/questions.",Would you have been willing to discuss your mental health with 

In [13]:
print(df_2014.shape)
print(df_2016.shape)
print(df_2017.shape)
print(df_2018.shape)
print(df_2019.shape)
print(df_2020.shape)
print(df_2021.shape)
print(df_2022.shape)
print(df_2023.shape)

(1259, 27)
(1433, 63)
(756, 123)
(417, 123)
(352, 82)
(180, 120)
(131, 124)
(164, 126)
(6, 126)


In [14]:
print(df_2014.columns)
print(df_2016.columns)
print(df_2017.columns)
print(df_2018.columns)
print(df_2019.columns)
print(df_2020.columns)
print(df_2021.columns)
print(df_2022.columns)
print(df_2023.columns)

Index(['Timestamp', 'Age', 'Gender', 'Country', 'state', 'self_employed',
       'family_history', 'treatment', 'work_interfere', 'no_employees',
       'remote_work', 'tech_company', 'benefits', 'care_options',
       'wellness_program', 'seek_help', 'anonymity', 'leave',
       'mental_health_consequence', 'phys_health_consequence', 'coworkers',
       'supervisor', 'mental_health_interview', 'phys_health_interview',
       'mental_vs_physical', 'obs_consequence', 'comments'],
      dtype='object')
Index(['Are you self-employed?',
       'How many employees does your company or organization have?',
       'Is your employer primarily a tech company/organization?',
       'Is your primary role within your company related to tech/IT?',
       'Does your employer provide mental health benefits as part of healthcare coverage?',
       'Do you know the options for mental health care available under your employer-provided coverage?',
       'Has your employer ever formally discussed mental 

### Data Cleaning & Preprocessing


##### Merged datasets from multiple years into a single dataframe and standardized column names. 
##### This step also included:
> ##### Handling missing values
> ##### Standardizing responses for key columns (e.g., gender, country)
> ##### Removing invalid ages
> ##### Creating consistent categorical features across years

In [15]:
column_mapping = {
    # -------- 2014 --------
    'Age': 'age',
    'Gender': 'gender',
    'Country': 'country',
    'self_employed': 'self_employed',
    'family_history': 'family_history',
    'treatment': 'treatment',
    'work_interfere': 'work_interfere',
    'no_employees': 'no_employees',
    'tech_company': 'tech_company',
    'benefits': 'benefits',
    'care_options': 'care_options',
    'wellness_program': 'wellness_program',
    'seek_help': 'seek_help',
    'anonymity': 'anonymity',
    'leave': 'leave',
    
    # -------- 2016 --------
    'Do you work remotely?':'remote_work',
    'What US state or territory do you live in?':'state',
    'What is your age?': 'age',
    'What is your gender?': 'gender',
    'What country do you live in?': 'country',
    'Are you self-employed?': 'self_employed',
    'Do you have a family history of mental illness?': 'family_history',
    'Have you ever sought treatment for a mental health issue from a mental health professional?': 'treatment',
    'How many employees does your company or organization have?': 'no_employees',
    'Is your employer primarily a tech company/organization?': 'tech_company',
    'Does your employer provide mental health benefits as part of healthcare coverage?': 'benefits',
    'Do you know the options for mental health care available under your employer-provided coverage?': 'care_options',
    'Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?': 'wellness_program',
    'Does your employer offer resources to learn more about mental health concerns and options for seeking help?': 'seek_help',
    'Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?': 'anonymity',
    'If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:': 'leave',
    'If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?': 'work_interfere',

    # -------- 2017+ --------
    'Does your employer provide mental health benefits as part of healthcare coverage?': 'benefits',
    'What US state or territory do you <strong>live</strong> in?':'state',
    '<strong>Are you self-employed?</strong>': 'self_employed',
    'Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?':'anonymity',
    'Have you ever sought treatment for a mental health disorder from a mental health professional?': 'treatment',
    'Do you know the options for mental health care available under your employer-provided health coverage?': 'care_options',
    'Does your employer offer resources to learn more about mental health disorders and options for seeking help?': 'seek_help',
    'If a mental health issue prompted you to request a medical leave from work, how easy or difficult would it be to ask for that leave?': 'leave',
    'Would you feel comfortable discussing a mental health issue with your direct supervisor(s)?': 'work_interfere',

    # -------- 2018+ --------
    'What country do you <strong>work</strong> in?': 'country',

    # -------- 2019 - 2023 variations --------
    'What US state or territory do you *live* in?':'state',
    'What country do you *live* in?':'country',
    '*Are you self-employed?*': 'self_employed',
    'Do you have a family history of mental illness?': 'family_history',
    'Have you ever been diagnosed with a mental health condition by a medical professional?': 'treatment',
    'How many people are employed at your company?': 'no_employees',
    'Is your company primarily a tech company?': 'tech_company',
    'Does your employer provide mental health benefits?': 'benefits',
    'Are you aware of the mental health options your employer provides?': 'care_options',
    'Does your employer discuss mental health in the workplace?': 'wellness_program',
    'Does your employer offer mental health resources?': 'seek_help',
    'Is your anonymity protected when using mental health resources?': 'anonymity',
    'How easy is it to take medical leave for mental health?': 'leave',
    'How does your mental health interfere with your work?': 'work_interfere',
    'Would you be willing to discuss a mental health issue with your coworkers?':'coworkers',
    'how much importance does your employer place on mental health?':'mental_health_consequence',
    'how much importance does your employer place on physical health?':'phys_health_consequence',
    'Would you feel comfortable discussing a mental health issue with your direct supervisor?':'supervisor',
    'Would you bring up your mental health with a potential employer in an interview?':'mental_health_interview',
    'Would you feel more comfortable talking to your coworkers about your physical health or your mental health?':'mental_vs_physical'
}

In [16]:
def clean_and_standardize(df, year, mapping, target_columns):
    df = df.rename(columns={k: v for k, v in mapping.items() if k in df.columns})
    df['survey_year'] = year
    available_cols = [col for col in target_columns if col in df.columns]
    return df[available_cols + ['survey_year']]

In [17]:
target_columns = [    
    'age', 'gender', 'country', 'state', 'self_employed', 'family_history',
     'treatment', 'work_interfere', 'no_employees', 'remote_work','tech_company', 'benefits', 'care_options', 'wellness_program',
     'seek_help', 'anonymity', 'leave', 'mental_health_consequence',
     'phys_health_consequence', 'coworkers', 'supervisor',
     'mental_vs_physical', 'obs_consequence', 'comments'
]

In [18]:
df_2014_cleaned = clean_and_standardize(df_2014, 2014, column_mapping, target_columns)
df_2016_cleaned = clean_and_standardize(df_2016, 2016, column_mapping, target_columns)
df_2017_cleaned = clean_and_standardize(df_2017, 2017, column_mapping, target_columns)
df_2018_cleaned = clean_and_standardize(df_2018, 2018, column_mapping, target_columns)
df_2019_cleaned = clean_and_standardize(df_2019, 2019, column_mapping, target_columns)
df_2020_cleaned = clean_and_standardize(df_2020, 2020, column_mapping, target_columns)
df_2021_cleaned = clean_and_standardize(df_2021, 2021, column_mapping, target_columns)
df_2022_cleaned = clean_and_standardize(df_2022, 2022, column_mapping, target_columns)
df_2023_cleaned = clean_and_standardize(df_2023, 2023, column_mapping, target_columns)

In [19]:
df_2014_cleaned

,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,remote_work,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_vs_physical,obs_consequence,comments,survey_year
0,37,Female,United States,IL,NaN,No,Yes,Often,6-25,No,Yes,Yes,Not sure,No,Yes,Yes,Somewhat easy,No,No,Some of them,Yes,Yes,No,NaN,2014
1,44,M,United States,IN,NaN,No,No,Rarely,More than 1000,No,No,Don't know,No,Don't know,Don't know,Don't know,Don't know,Maybe,No,No,No,Don't know,No,NaN,2014
2,32,Male,Canada,NaN,NaN,No,No,Rarely,6-25,No,Yes,No,No,No,No,Don't know,Somewhat difficult,No,No,Yes,Yes,No,No,NaN,2014
3,31,Male,United Kingdom,NaN,NaN,Yes,Yes,Often,26-100,No,Yes,No,Yes,No,No,No,Somewhat difficult,Yes,Yes,Some of them,No,No,Yes,NaN,2014
4,31,Male,United States,TX,NaN,No,No,Never,100-500,Yes,Yes,Yes,No,Don't know,Don't know,Don't know,Don't know,No,No,Some of them,Yes,Don't know,No,NaN,2014
5,33,Male,United States,TN,NaN,Yes,No,Sometimes,6-25,No,Yes,Yes,Not sure,No,Don't know,Don't know,Don't know,No,No,Yes,Yes,Don't know,No,NaN,2014
6,35,Female,United States,MI,NaN,Yes,Yes,Sometimes,1-5,Yes,Yes,No,No,No,No,No,Somewhat difficult,Maybe,Maybe,Some of them,No,Don't know,No,NaN,2014
7,39,M,Canada,NaN,NaN,No,No,Never,1-5,Yes,Yes,No,Yes,No,No,Yes,Don't know,No,No,No,No,No,No,NaN,2014
8,42,Female,United States,IL,NaN,Yes,Yes,Sometimes,100-500,No,Yes,Yes,Yes,No,No,No,Very difficult,Maybe,No,Yes,Yes,No,No,NaN,2014
9,23,Male,Canada,NaN,NaN,No,No,Never,26-100,No,Yes,Don't know,No,Don't know,Don't know,Don't know,Don't know,No,No,Yes,Yes,Yes,No,NaN,2014


In [20]:
df_2016_cleaned

,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,remote_work,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,survey_year
0,39,Male,United Kingdom,NaN,0,No,0,Not applicable to me,26-100,Sometimes,1.0,Not eligible for coverage / N/A,NaN,No,No,I don't know,Very easy,2016
1,29,male,United States of America,Illinois,0,Yes,1,Sometimes,6-25,Never,1.0,No,Yes,Yes,Yes,Yes,Somewhat easy,2016
2,38,Male,United Kingdom,NaN,0,No,1,Not applicable to me,6-25,Always,1.0,No,NaN,No,No,I don't know,Neither easy nor difficult,2016
3,43,male,United Kingdom,NaN,1,No,1,Sometimes,NaN,Sometimes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016
4,43,Female,United States of America,Illinois,0,Yes,1,Sometimes,6-25,Sometimes,0.0,Yes,Yes,No,No,No,Neither easy nor difficult,2016
5,42,Male,United Kingdom,NaN,0,No,1,Often,More than 1000,Sometimes,1.0,Yes,I am not sure,No,Yes,Yes,Somewhat easy,2016
6,30,M,United States of America,Tennessee,0,No,0,Not applicable to me,26-100,Sometimes,1.0,I don't know,No,No,No,I don't know,Somewhat easy,2016
7,37,female,United States of America,Virginia,0,Yes,1,Often,More than 1000,Always,1.0,Yes,Yes,No,Yes,Yes,Very easy,2016
8,44,Female,United States of America,California,0,Yes,1,Often,26-100,Sometimes,0.0,I don't know,No,No,No,I don't know,Very difficult,2016
9,30,Male,United States of America,Kentucky,1,Yes,1,Often,NaN,Always,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016


In [21]:
df_2017_cleaned

,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,tech_company,care_options,wellness_program,seek_help,anonymity,leave,mental_vs_physical,survey_year
0,27.0,Female,United Kingdom,NaN,0,No,1,Yes,100-500,1.0,Yes,No,I don't know,I don't know,I don't know,Same level of comfort for each,2017
1,31.0,male,United Kingdom,NaN,0,No,0,Maybe,100-500,1.0,Yes,No,No,I don't know,I don't know,Same level of comfort for each,2017
2,36.0,male,United States of America,Missouri,0,Yes,1,Yes,6-25,1.0,No,I don't know,No,Yes,Difficult,Same level of comfort for each,2017
3,22.0,Male,United States of America,Washington,0,I don't know,1,Yes,More than 1000,1.0,Yes,I don't know,I don't know,Yes,Difficult,Same level of comfort for each,2017
4,52.0,female,United States of America,Illinois,1,Yes,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017
5,30.0,male,United States of America,California,0,Yes,0,Maybe,100-500,1.0,No,No,I don't know,Yes,Somewhat easy,Physical health,2017
6,36.0,F,United States of America,Washington,0,Yes,1,Yes,6-25,1.0,Yes,No,No,Yes,Very easy,Same level of comfort for each,2017
7,38.0,Female,United States of America,Georgia,0,Yes,1,Yes,26-100,1.0,No,No,No,I don't know,Somewhat easy,Physical health,2017
8,35.0,Male,Switzerland,NaN,0,I don't know,0,Maybe,100-500,0.0,No,No,No,Yes,Very easy,Same level of comfort for each,2017
9,36.0,male,India,NaN,1,No,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017


In [22]:
df_2018_cleaned

,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_vs_physical,survey_year
0,57,Female,Canada,NaN,0,Yes,1,No,More than 1000,1.0,Yes,Yes,Yes,Yes,Yes,Somewhat difficult,Physical health,2018
1,29,male,United States of America,Massachusetts,0,Yes,1,No,More than 1000,1.0,Yes,Yes,No,I don't know,I don't know,Somewhat difficult,Physical health,2018
2,46,Male,United States of America,Florida,0,Yes,0,No,6-25,0.0,Yes,Yes,No,No,I don't know,Somewhat easy,Physical health,2018
3,34,male,Norway,NaN,0,No,0,No,6-25,1.0,No,No,No,No,I don't know,Neither easy nor difficult,Physical health,2018
4,29,Ostensibly Male,United States of America,Tennessee,0,Yes,1,Yes,26-100,1.0,Yes,Yes,Yes,Yes,Yes,Somewhat easy,Same level of comfort for each,2018
5,55,male,United States of America,South Carolina,0,No,0,Yes,100-500,1.0,Yes,Yes,No,I don't know,I don't know,Somewhat easy,Same level of comfort for each,2018
6,29,Agender,Finland,NaN,0,No,1,No,More than 1000,1.0,Yes,Yes,No,No,I don't know,I don't know,Physical health,2018
7,35,male,Poland,NaN,0,I don't know,1,No,26-100,1.0,Not eligible for coverage / NA,NaN,No,No,I don't know,I don't know,Physical health,2018
8,33,"male, born with xy chromosoms",Russia,NaN,0,Yes,0,No,More than 1000,1.0,I don't know,No,No,No,No,I don't know,Physical health,2018
9,37,Male,United States of America,Iowa,0,No,0,No,More than 1000,0.0,Yes,No,No,Yes,I don't know,Somewhat difficult,Physical health,2018


In [23]:
df_2019_cleaned

,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_vs_physical,survey_year
0,25,Male,United States of America,Nebraska,False,No,False,Yes,26-100,True,I don't know,No,Yes,Yes,I don't know,Very easy,Physical health,2019
1,51,male,United States of America,Nebraska,False,Yes,False,Maybe,26-100,True,Yes,No,No,Yes,Yes,I don't know,Physical health,2019
2,27,Male,United States of America,Illinois,False,I don't know,False,No,26-100,True,I don't know,No,No,I don't know,I don't know,Somewhat difficult,Same level of comfort for each,2019
3,37,male,United States of America,Nebraska,False,Yes,False,Yes,100-500,True,I don't know,No,Yes,Yes,Yes,Very easy,Physical health,2019
4,46,m,United States of America,Nebraska,False,No,False,No,26-100,True,I don't know,No,I don't know,I don't know,I don't know,I don't know,Physical health,2019
5,36,female,United States of America,Nebraska,False,Yes,True,No,100-500,True,Yes,No,No,No,I don't know,Somewhat difficult,Physical health,2019
6,39,Female,United States of America,Nebraska,False,Yes,True,Maybe,26-100,True,Yes,Yes,No,I don't know,Yes,Somewhat easy,Physical health,2019
7,35,Male,United States of America,Wisconsin,True,Yes,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019
8,49,NaN,United Kingdom,NaN,False,No,False,Yes,26-100,True,Yes,No,Yes,Yes,Yes,Very easy,Same level of comfort for each,2019
9,45,Male,United Kingdom,NaN,False,I don't know,True,Yes,6-25,True,I don't know,NaN,No,I don't know,I don't know,Somewhat easy,Same level of comfort for each,2019


In [24]:
df_2020_cleaned

,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_vs_physical,survey_year
0,45,Male,United States of America,Connecticut,1,Yes,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020
1,24,female,Russia,NaN,1,Yes,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020
2,46,Male,India,NaN,1,Yes,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020
3,25,Female,Canada,NaN,1,Yes,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020
4,25,F,Canada,NaN,1,I don't know,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020
5,5,f,United States of America,Alabama,1,No,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020
6,1,NaN,Japan,NaN,1,Yes,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020
7,35,Male,United States of America,Minnesota,1,Yes,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020
8,34,female,Ireland,NaN,1,No,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020
9,23,Female,Turkey,NaN,1,No,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020


In [25]:
df_2021_cleaned

,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_vs_physical,survey_year
0,28,Female,United States of America,Alaska,0,I don't know,0,Maybe,26-100,1.0,I don't know,No,No,I don't know,I don't know,Very easy,Physical health,2021
1,41,male,Brazil,NaN,0,No,0,No,500-1000,1.0,Yes,No,Yes,Yes,Yes,I don't know,Physical health,2021
2,35,Male,Brazil,NaN,0,No,0,Maybe,100-500,1.0,Yes,Yes,No,I don't know,I don't know,Somewhat easy,Same level of comfort for each,2021
3,20,male,Italy,NaN,1,Yes,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021
4,35,female,Canada,NaN,0,No,1,No,More than 1000,0.0,Yes,No,Yes,Yes,I don't know,Difficult,Physical health,2021
5,30,Female,United States of America,New York,0,No,0,No,6-25,1.0,No,No,No,No,No,Difficult,Same level of comfort for each,2021
6,34,female,United States of America,Maryland,0,No,0,Yes,More than 1000,1.0,No,Yes,Yes,Yes,Yes,Somewhat easy,Physical health,2021
7,23,female,Germany,NaN,0,I don't know,0,Maybe,More than 1000,0.0,Yes,Yes,No,No,I don't know,Very easy,Physical health,2021
8,19,Female,India,NaN,0,No,0,No,More than 1000,1.0,No,No,Yes,No,No,Somewhat difficult,Same level of comfort for each,2021
9,34,f,Belarus,NaN,1,Yes,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021


In [26]:
df_2022_cleaned

,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_vs_physical,survey_year
0,23,female,Russia,NaN,1,I don't know,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022
1,43,Male,Netherlands,NaN,0,Yes,1,Yes,26-100,1.0,I don't know,No,No,No,I don't know,Very easy,Physical health,2022
2,38,male,Sweden,NaN,0,Yes,1,Yes,26-100,1.0,No,Yes,I don't know,I don't know,Yes,Somewhat easy,Physical health,2022
3,35,Female,United States of America,Oregon,0,Yes,1,Maybe,More than 1000,1.0,Yes,Yes,Yes,Yes,I don't know,I don't know,Physical health,2022
4,33,Female,United Kingdom,NaN,0,Yes,1,Yes,26-100,1.0,Yes,No,Yes,Yes,Yes,Very easy,Same level of comfort for each,2022
5,26,female,Argentina,NaN,0,Yes,1,Yes,26-100,1.0,No,Yes,No,No,No,Very easy,Same level of comfort for each,2022
6,29,Male,United States of America,California,0,No,0,No,26-100,1.0,I don't know,No,No,No,I don't know,Neither easy nor difficult,Physical health,2022
7,45,F,Austria,NaN,1,No,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022
8,34,F,France,NaN,0,No,1,No,100-500,1.0,Yes,Yes,Yes,Yes,Yes,Somewhat easy,Physical health,2022
9,35,Female,Canada,NaN,1,I don't know,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022


In [27]:
df_2023_cleaned

,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_vs_physical,survey_year
0,40,Female,United States of America,Colorado,0,I don't know,1,Maybe,26-100,1.0,No,No,No,No,Yes,Somewhat easy,Mental health,2023
1,36,Male,United States of America,Indiana,0,No,1,No,6-25,1.0,I don't know,No,No,I don't know,I don't know,I don't know,Physical health,2023
2,44,male,United States of America,Maryland,1,Yes,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023
3,53,Female,United States of America,Indiana,0,Yes,1,Maybe,500-1000,0.0,Yes,Yes,Yes,No,I don't know,Neither easy nor difficult,Same level of comfort for each,2023
4,62,male,United States of America,Ohio,1,No,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023
5,39,Male,Canada,NaN,0,Yes,1,No,More than 1000,1.0,I don't know,No,Yes,I don't know,Yes,Somewhat easy,Mental health,2023


In [28]:
print(df_2014_cleaned.shape)
print(df_2016_cleaned.shape)
print(df_2017_cleaned.shape)
print(df_2018_cleaned.shape)
print(df_2019_cleaned.shape)
print(df_2020_cleaned.shape)
print(df_2021_cleaned.shape)
print(df_2022_cleaned.shape)
print(df_2023_cleaned.shape)

(1259, 25)
(1433, 18)
(756, 17)
(417, 18)
(352, 18)
(180, 18)
(131, 18)
(164, 18)
(6, 18)


In [29]:
for i, df in zip([2014, 2016, 2017, 2018, 2019,2020, 2021, 2022, 2023],
                 [df_2014_cleaned, df_2016_cleaned, df_2017_cleaned, df_2018_cleaned, df_2019_cleaned, df_2020_cleaned, df_2021_cleaned, df_2022_cleaned, df_2023_cleaned]):
    missing = set(target_columns) - set(df.columns)
    print(f"Year {i} missing columns: {missing}")

Year 2014 missing columns: set()
Year 2016 missing columns: {'supervisor', 'mental_vs_physical', 'obs_consequence', 'coworkers', 'comments', 'mental_health_consequence', 'phys_health_consequence'}
Year 2017 missing columns: {'supervisor', 'benefits', 'obs_consequence', 'remote_work', 'coworkers', 'comments', 'mental_health_consequence', 'phys_health_consequence'}
Year 2018 missing columns: {'supervisor', 'obs_consequence', 'remote_work', 'coworkers', 'comments', 'mental_health_consequence', 'phys_health_consequence'}
Year 2019 missing columns: {'supervisor', 'obs_consequence', 'remote_work', 'coworkers', 'comments', 'mental_health_consequence', 'phys_health_consequence'}
Year 2020 missing columns: {'supervisor', 'obs_consequence', 'remote_work', 'coworkers', 'comments', 'mental_health_consequence', 'phys_health_consequence'}
Year 2021 missing columns: {'supervisor', 'obs_consequence', 'remote_work', 'coworkers', 'comments', 'mental_health_consequence', 'phys_health_consequence'}
Year 2

In [30]:
merged_df_all_years = pd.concat([
    df_2014_cleaned,
    df_2016_cleaned,
    df_2017_cleaned,
    df_2018_cleaned,
    df_2019_cleaned,
    df_2020_cleaned,
    df_2021_cleaned,
    df_2022_cleaned,
    df_2023_cleaned
    ], ignore_index=True)


print("Final merged shape:", merged_df_all_years.shape)
merged_df_all_years.tail()

Final merged shape: (4698, 25)


,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,remote_work,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_vs_physical,obs_consequence,comments,survey_year
4693,36.0,Male,United States of America,Indiana,0,No,1,No,6-25,NaN,1.0,I don't know,No,No,I don't know,I don't know,I don't know,NaN,NaN,NaN,NaN,Physical health,NaN,NaN,2023
4694,44.0,male,United States of America,Maryland,1,Yes,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023
4695,53.0,Female,United States of America,Indiana,0,Yes,1,Maybe,500-1000,NaN,0.0,Yes,Yes,Yes,No,I don't know,Neither easy nor difficult,NaN,NaN,NaN,NaN,Same level of comfort for each,NaN,NaN,2023
4696,62.0,male,United States of America,Ohio,1,No,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023
4697,39.0,Male,Canada,NaN,0,Yes,1,No,More than 1000,NaN,1.0,I don't know,No,Yes,I don't know,Yes,Somewhat easy,NaN,NaN,NaN,NaN,Mental health,NaN,NaN,2023


In [31]:
round(merged_df_all_years.describe(include='all'),2)

,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,remote_work,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_vs_physical,obs_consequence,comments,survey_year
count,4.696000e+03,4664,4696,2779,4680.0,4698,4698.0,4133,4110,2692,4110.0,3467,3803,4110,4110,4110,4110,1259,1259,1259,1259,2964,1259,164,4698.00
unique,NaN,187,95,93,4.0,3,4.0,8,6,5,4.0,6,4,4,4,4,8,3,3,3,3,6,2,160,NaN
top,NaN,Male,United States of America,California,0.0,Yes,1.0,Sometimes,More than 1000,No,1.0,Yes,No,No,No,I don't know,Somewhat easy,No,No,Some of them,Yes,Physical health,No,* Small family business - YMMV.,NaN
freq,NaN,2052,2044,267,2851.0,2026,2004.0,828,1064,883,2115.0,1571,1671,2663,1905,1756,1004,490,925,774,516,1148,1075,5,NaN
mean,2.129475e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016.54
std,1.459271e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.15
min,-1.726000e+03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014.00
25%,2.800000e+01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014.00
50%,3.300000e+01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016.00
75%,3.900000e+01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018.00


##### Age has negative values, values over 1000
##### Gender has too many categories (187 unique)

In [32]:
merged_df_all_years.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4698 entries, 0 to 4697
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   age                        4696 non-null   float64
 1   gender                     4664 non-null   object 
 2   country                    4696 non-null   object 
 3   state                      2779 non-null   object 
 4   self_employed              4680 non-null   object 
 5   family_history             4698 non-null   object 
 6   treatment                  4698 non-null   object 
 7   work_interfere             4133 non-null   object 
 8   no_employees               4110 non-null   object 
 9   remote_work                2692 non-null   object 
 10  tech_company               4110 non-null   object 
 11  benefits                   3467 non-null   object 
 12  care_options               3803 non-null   object 
 13  wellness_program           4110 non-null   objec

In [33]:
merged_df_all_years.dtypes

age                          float64
gender                        object
country                       object
state                         object
self_employed                 object
family_history                object
treatment                     object
work_interfere                object
no_employees                  object
remote_work                   object
tech_company                  object
benefits                      object
care_options                  object
wellness_program              object
seek_help                     object
anonymity                     object
leave                         object
mental_health_consequence     object
phys_health_consequence       object
coworkers                     object
supervisor                    object
mental_vs_physical            object
obs_consequence               object
comments                      object
survey_year                    int64
dtype: object

In [34]:
merged_df_all_years.isna().sum().sort_values(ascending=False).reset_index()

,index,0
0,comments,4534
1,coworkers,3439
2,supervisor,3439
3,phys_health_consequence,3439
4,obs_consequence,3439
5,mental_health_consequence,3439
6,remote_work,2006
7,state,1919
8,mental_vs_physical,1734
9,benefits,1231


In [35]:
x=round((merged_df_all_years.isna().sum()/merged_df_all_years.shape[0])*100,2)
x = pd.DataFrame(x)
x = x.sort_values(by=x.columns[0], ascending=False).reset_index()
x

,index,0
0,comments,96.51
1,coworkers,73.20
2,supervisor,73.20
3,phys_health_consequence,73.20
4,obs_consequence,73.20
5,mental_health_consequence,73.20
6,remote_work,42.70
7,state,40.85
8,mental_vs_physical,36.91
9,benefits,26.20


##### Comments column has almost 96% null values; it is not a mandatory question in the survey. Can be dropped.
> ##### May keep for sentiment analysis later. TBD, but holding in for now
##### Work_interfere and self_employed will need to be treated
##### Null values are observed. Most come from the State column which does not contribute to the research and can be dropped.

In [36]:
merged_df_all_years['state'].unique()

array(['IL', 'IN', nan, 'TX', 'TN', 'MI', 'OH', 'CA', 'CT', 'MD', 'NY',
       'NC', 'MA', 'IA', 'PA', 'WA', 'WI', 'UT', 'NM', 'OR', 'FL', 'MN',
       'MO', 'AZ', 'CO', 'GA', 'DC', 'NE', 'WV', 'OK', 'KS', 'VA', 'NH',
       'KY', 'AL', 'NV', 'NJ', 'SC', 'VT', 'SD', 'ID', 'MS', 'RI', 'WY',
       'LA', 'ME', 'Illinois', 'Tennessee', 'Virginia', 'California',
       'Kentucky', 'Oregon', 'Pennsylvania', 'New Jersey', 'Georgia',
       'Washington', 'New York', 'Indiana', 'Minnesota', 'West Virginia',
       'Florida', 'Massachusetts', 'North Dakota', 'Texas', 'Maryland',
       'Wisconsin', 'Michigan', 'Vermont', 'North Carolina', 'Kansas',
       'District of Columbia', 'Nevada', 'Utah', 'Connecticut',
       'Colorado', 'Ohio', 'Iowa', 'South Dakota', 'Nebraska', 'Maine',
       'Missouri', 'Arizona', 'Oklahoma', 'Idaho', 'Rhode Island',
       'Alabama', 'Louisiana', 'South Carolina', 'New Hampshire',
       'New Mexico', 'Montana', 'Alaska', 'Delaware', 'Wyoming'],
      dtype=objec

In [37]:
merged_df_all_years['country'].value_counts()

country
United States of America    2044
United States                751
United Kingdom               507
Canada                       225
Germany                      154
Netherlands                  110
India                        110
Australia                     83
France                        54
Brazil                        52
Ireland                       52
Spain                         35
Sweden                        32
Switzerland                   29
Portugal                      26
Poland                        26
Italy                         25
New Zealand                   25
South Africa                  21
Russia                        21
Belgium                       17
Mexico                        14
Finland                       14
Bulgaria                      14
Austria                       14
Norway                        12
Israel                        12
Denmark                       10
Indonesia                     10
Romania                       10
Ja

##### Nearly 60% of responses are from the US
##### The other 40% is spread out across several countries

In [38]:
merged_df_all_years = merged_df_all_years.drop(['state'], axis=1)

In [39]:
# Function to observe unique categorical values for each variable.
def see_unique_categories(df):
    """
    Prints the unique categories of each categorical variable in the DataFrame.
    """
    for col in merged_df_all_years.select_dtypes(include=['object', 'category']).columns:
        print(f"--- {col} ---")
        print(merged_df_all_years[col].unique())
        print()
        print("------------------------------------------------------------------------------")

In [40]:
see_unique_categories(merged_df_all_years)

--- gender ---
['Female' 'M' 'Male' 'male' 'female' 'm' 'Male-ish' 'maile' 'Trans-female'
 'Cis Female' 'F' 'something kinda male?' 'Cis Male' 'Woman' 'f' 'Mal'
 'Male (CIS)' 'queer/she/they' 'non-binary' 'Femake' 'woman' 'Make' 'Nah'
 'All' 'Enby' 'fluid' 'Genderqueer' 'Female ' 'Androgyne' 'Agender'
 'cis-female/femme' 'Guy (-ish) ^_^' 'male leaning androgynous' 'Male '
 'Man' 'Trans woman' 'msle' 'Neuter' 'Female (trans)' 'queer'
 'Female (cis)' 'Mail' 'cis male' 'A little about you' 'Malr' 'p' 'femail'
 'Cis Man' 'ostensibly male, unsure what that really means'
 'I identify as female.' 'female ' 'Bigender' 'Female assigned at birth '
 'man' 'fm' 'Cis female ' 'Transitioned, M2F' 'Genderfluid (born female)'
 'Other/Transfeminine' 'Female or Multi-Gender Femme' 'female/woman'
 'Cis male' 'Male.' 'Androgynous' 'male 9:1 female, roughly' nan
 'Male (cis)' 'Other' 'nb masculine' 'Cisgender Female' 'Sex is male'
 'none of your business' 'genderqueer' 'Human' 'Genderfluid'
 'genderqueer w

In [41]:
merged_df_all_years['gender'].value_counts().reset_index()

,gender,count
0,Male,2052
1,male,747
2,Female,572
3,female,319
4,M,276
5,m,166
6,F,124
7,f,69
8,Male,25
9,Female,21


In [42]:
male_keywords = ['mostly male', 'cisgender male','MAle', 'male/he/him','CIS Male','Cisgender male','Let\'s keep it simple and say "male"',
                 'Identify as male', 'Malel', 'dude', 'Ostensibly Male', 'male, born with xy chromosoms','Male, cis' ,'cis male ','Cis-male',
                 "male (hey this is the tech industry you're talking about)",'cis hetero male' ,'cis man' ,'MALE' ,'cis-male','mail', 'M|','male ' ,
                 "I'm a man why didn't you make this a drop down question. You should of asked sex? And I would of answered yes please. Seriously how much text can this take? ",
                 'Dude' ,'Male (cis)', 'Sex is male', 'Cis male', 'Male.', 'man', 'ostensibly male','Cis Man' ,'Malr', 'cis male', 'Mail', 
                 'Man','Make','Male (CIS)','Mal','Cis Male','Male','male', 'Male-ish', 'maile', 'm','M', 'man', 'cis male', 'cis man', 'male ',
                 'msle', 'malr', 'mal','maile', 'mail', 'make', 'guy', 'male leaning androgynous']

female_keywords = ['Femile' ,'FEMALE' ,'female, she/her','Female-identified' ,'cis woman','femmina','cisgender female' ,'Cisgendered woman',
                   'Female,cis-gendered','Female/gender non-binary.' ,'Cis woman', 'Female (cisgender)', 'Cis-Female','I identify as female' ,
                   '*shrug emoji* (F)', 'cis female' , 'F, cisgender' ,'Female-ish' ,'female (cisgender)' ,'Female (cis) ', 'Woman-identified',
                   'cis-Female','female (cis)','My sex is female.','femalw','Cis-woman' ,'Female (props for making this a freeform field, though)' ,
                   ' Female','fem','Cisgender Female','female/woman', 'fm' ,'Cis female ','I identify as female.','female ','Female assigned at birth ',
                   'femail', 'Female (cis)','Femake','Woman','Cis Female' ,'F','Female','female', 'f', 'woman', 'femake', 'femail', 'cis female',
                   'cis-female', 'female ', 'cis female/femme', 'female (cis)']

In [43]:
def general_classifier(value):
   
    val_norm = str(value).strip().lower()
    
    if any(kw in val_norm for kw in female_keywords):
        return 'Female'
    elif any(kw in val_norm for kw in male_keywords):
        return 'Male'
    else:
        return 'Other'


In [44]:
merged_df_all_years['gender'] = merged_df_all_years['gender'].apply(general_classifier)

In [45]:
merged_df_all_years['gender'].value_counts().reset_index()

,gender,count
0,Male,3375
1,Female,1225
2,Other,98


##### Gender consolidated to 2 values for simplicity
##### A dropdown in further surveys would make this cleaner and allow for broader inclusivity

In [46]:
see_unique_categories(merged_df_all_years)

--- gender ---
['Female' 'Male' 'Other']

------------------------------------------------------------------------------
--- country ---
['United States' 'Canada' 'United Kingdom' 'Bulgaria' 'France' 'Portugal'
 'Netherlands' 'Switzerland' 'Poland' 'Australia' 'Germany' 'Russia'
 'Mexico' 'Brazil' 'Slovenia' 'Costa Rica' 'Austria' 'Ireland' 'India'
 'South Africa' 'Italy' 'Sweden' 'Colombia' 'Latvia' 'Romania' 'Belgium'
 'New Zealand' 'Zimbabwe' 'Spain' 'Finland' 'Uruguay' 'Israel'
 'Bosnia and Herzegovina' 'Hungary' 'Singapore' 'Japan' 'Nigeria'
 'Croatia' 'Norway' 'Thailand' 'Denmark' 'Bahamas, The' 'Greece' 'Moldova'
 'Georgia' 'China' 'Czech Republic' 'Philippines'
 'United States of America' 'Lithuania' 'Venezuela' 'Argentina' 'Vietnam'
 'Slovakia' 'Bangladesh' 'Algeria' 'Pakistan' 'Afghanistan' 'Other'
 'Brunei' 'Iran' 'Ecuador' 'Chile' 'Guatemala' 'Taiwan' 'Serbia' 'Estonia'
 'Iceland' 'Indonesia' 'Jordan' 'Eritrea' 'Swaziland' 'Ukraine'
 'Luxembourg' 'Turkey' nan 'Mauritius' 'S

In [47]:
merged_df_all_years['age'].value_counts().reset_index()

,age,count
0,3.000000e+01,275
1,3.200000e+01,247
2,2.900000e+01,243
3,3.100000e+01,237
4,2.800000e+01,232
5,3.400000e+01,224
6,3.300000e+01,218
7,3.500000e+01,217
8,2.700000e+01,213
9,2.600000e+01,207


In [48]:
merged_df_all_years = merged_df_all_years[(merged_df_all_years['age'] >= 18) & (merged_df_all_years['age'] <= 80)]

In [49]:
merged_df_all_years['age'].value_counts().reset_index()

,age,count
0,30.0,275
1,32.0,247
2,29.0,243
3,31.0,237
4,28.0,232
5,34.0,224
6,33.0,218
7,35.0,217
8,27.0,213
9,26.0,207


##### Outliers for the age variable are removed (negative ages and exaggeratedly high values).
> ##### Outliers occur in only two observations and dropping will not greatly impact the data
> ##### Since the Age variable was improperly answered, its possible other fields are similarly unreliable. Dropping may increase robustness of the data

### Feature Engineering

##### Creating `support_score` based on support-related columns
##### Binning age into age groups

In [50]:
bins = [18, 25, 35, 45, 55, 65, 80]
labels = ['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
merged_df_all_years['age_group'] = pd.cut(merged_df_all_years['age'], bins=bins, labels=labels, right=False)

In [51]:
merged_df_all_years['age_group'].value_counts().reset_index()

,age_group,count
0,25-34,2263
1,35-44,1424
2,18-24,467
3,45-54,403
4,55-64,107
5,65+,13


In [52]:
for col in merged_df_all_years.select_dtypes(include=['object', 'category','float64', 'int64']).columns:
    print(f"--- {col} ---")
    print(merged_df_all_years[col].unique())
    print()
    print("------------------------------------------------------------------------------")

--- age ---
[37. 44. 32. 31. 33. 35. 39. 42. 23. 29. 36. 27. 46. 41. 34. 30. 40. 38.
 50. 24. 18. 28. 26. 22. 19. 25. 45. 21. 43. 56. 60. 54. 55. 48. 20. 57.
 58. 47. 62. 51. 65. 49. 53. 61. 72. 52. 63. 66. 59. 74. 70. 64. 67. 76.]

------------------------------------------------------------------------------
--- gender ---
['Female' 'Male' 'Other']

------------------------------------------------------------------------------
--- country ---
['United States' 'Canada' 'United Kingdom' 'Bulgaria' 'France' 'Portugal'
 'Netherlands' 'Switzerland' 'Poland' 'Australia' 'Germany' 'Russia'
 'Mexico' 'Brazil' 'Slovenia' 'Costa Rica' 'Austria' 'Ireland' 'India'
 'South Africa' 'Italy' 'Sweden' 'Colombia' 'Latvia' 'Romania' 'Belgium'
 'New Zealand' 'Spain' 'Finland' 'Uruguay' 'Israel'
 'Bosnia and Herzegovina' 'Hungary' 'Singapore' 'Japan' 'Nigeria'
 'Croatia' 'Norway' 'Thailand' 'Denmark' 'Greece' 'Moldova' 'Georgia'
 'China' 'Czech Republic' 'Philippines' 'United States of America'
 'Lithuan

##### Encoding responses

In [53]:
merged_df_all_years.isna().sum().sort_values(ascending=False).reset_index()

,index,0
0,comments,4516
1,coworkers,3426
2,phys_health_consequence,3426
3,mental_health_consequence,3426
4,obs_consequence,3426
5,supervisor,3426
6,remote_work,1998
7,mental_vs_physical,1723
8,benefits,1224
9,care_options,887


In [54]:
# Helper functions

def map_binary(value):
    """Map binary-like values to 0/1. Handles Yes/No, True/False, 1/0."""
    if pd.isna(value):
        return np.nan
    val = str(value).strip().lower()
    if val in ['yes','y','1','1.0','true','t']:
        return 1
    elif val in ['no','n','0','0.0','false','f']:
        return 0
    else:
        return np.nan

def map_ternary(value):
    """Map ternary values: Yes=1, No=0, Don't know / Not sure / Not eligible = 2"""
    if pd.isna(value):
        return np.nan
    val = str(value).strip().lower()
    if val in ['yes','y','1','1.0','true','t']:
        return 1
    elif val in ['no','n','0','0.0','false','f']:
        return 0
    else:
        return 2

def normalize_binary_column(col):
    """Normalize a column to canonical 0/1/NaN"""
    return col.map(map_binary)

In [55]:
# Main Encoding Function

def encode_survey_clean_final(
    df,
    ordinal_mappings=None,
    binary_candidates=None,
    nominal_onehot=None,
    exclude_columns=None,
    verbose=True
):
    """
    Fully robust survey encoding:
    - Binary candidates normalized to 0/1 (_enc column added)
    - Ordinals mapped according to provided mapping
    - Ternary detected automatically (Yes/No/Don't know)
    - Nominal one-hot encoding for multi-category variables
    - Preserves original columns
    - Excludes free-text columns
    """

    df_encoded = df.copy()
    binary_candidates = binary_candidates or []
    nominal_onehot = nominal_onehot or []
    exclude_columns = exclude_columns or []

    # ---------------- Step 1: Normalize binary candidates ----------------
    for col in binary_candidates:
        if col in df_encoded.columns:
            df_encoded[col] = normalize_binary_column(df_encoded[col])
            df_encoded[f"{col}_enc"] = df_encoded[col]  # Add _enc column
            if verbose:
                print(f"Binary candidate normalized and encoded → {col} → {col}_enc")

    # ---------------- Step 2: Encode remaining columns ----------------
    for col in df_encoded.columns:
        if col in exclude_columns:
            if verbose:
                print(f"Excluded → {col}")
            continue

        # Skip numeric columns (binary candidates already handled)
        if pd.api.types.is_numeric_dtype(df_encoded[col]):
            continue

        new_col = f"{col}_enc"

        # Normalize strings for detection
        col_str = df_encoded[col].astype(str).str.strip().str.lower()
        unique_vals = col_str.replace('nan', np.nan).dropna().unique()
        sample_vals = [str(v) for v in unique_vals]

        # ----- Ordinal mapping -----
        if ordinal_mappings and col in ordinal_mappings:
            df_encoded[new_col] = df_encoded[col].map(ordinal_mappings[col])
            if verbose:
                print(f"Ordinal → {col} mapped to {new_col}")
            continue

        # ----- Nominal one-hot override -----
        if col in nominal_onehot:
            dummies = pd.get_dummies(df_encoded[col], prefix=new_col, dummy_na=False)
            df_encoded = pd.concat([df_encoded, dummies], axis=1)
            if verbose:
                print(f"Nominal one-hot → {col} expanded to {list(dummies.columns)}")
            continue

        # ----- Ternary detection (Yes/No/Don't know) -----
        # Only trigger if column is not already binary candidate
        if col not in binary_candidates:
            if any(['don' in v or 'not sure' in v or 'not eligible' in v for v in sample_vals]):
                df_encoded[new_col] = col_str.map(map_ternary)
                if verbose:
                    print(f"Ternary → {col} mapped to {new_col} (0/1/2)")
                continue

        # ----- Otherwise → one-hot multi-category -----
        dummies = pd.get_dummies(df_encoded[col], prefix=new_col, dummy_na=False)
        df_encoded = pd.concat([df_encoded, dummies], axis=1)
        if verbose:
            print(f"One-hot → {col} expanded to {list(dummies.columns)}")

    return df_encoded

In [56]:
# Binary columns
binary_candidates = ['self_employed','treatment','tech_company','obs_consequence']

# Ordinal mappings
ordinal_mappings = {
    'work_interfere': {'Never':0, 'Rarely':1, 'Sometimes':2, 'Often':3},
    'leave': {'Very easy':0, 'Somewhat easy':1, 'Neither easy nor difficult':2,
              'Somewhat difficult':3, 'Difficult':3, 'Very difficult':4}
}

# Nominal one-hot columns (multi-category)
nominal_onehot = ['no_employees','remote_work','age_group',
                  'mental_health_consequence','phys_health_consequence',
                  'coworkers','supervisor']

# Columns to exclude (free-text)
exclude_columns = ['comments','country']

# Country stays as a single column
# Optionally encode separately if needed:
# df['country_enc'] = df['country']

# Encode dataset
df_clean = encode_survey_clean_final(
    merged_df_all_years,
    ordinal_mappings=ordinal_mappings,
    binary_candidates=binary_candidates,
    nominal_onehot=nominal_onehot,
    exclude_columns=exclude_columns,
    verbose=True
)

Binary candidate normalized and encoded → self_employed → self_employed_enc
Binary candidate normalized and encoded → treatment → treatment_enc
Binary candidate normalized and encoded → tech_company → tech_company_enc
Binary candidate normalized and encoded → obs_consequence → obs_consequence_enc
One-hot → gender expanded to ['gender_enc_Female', 'gender_enc_Male', 'gender_enc_Other']
Excluded → country
Ternary → family_history mapped to family_history_enc (0/1/2)
Ordinal → work_interfere mapped to work_interfere_enc
Nominal one-hot → no_employees expanded to ['no_employees_enc_1-5', 'no_employees_enc_100-500', 'no_employees_enc_26-100', 'no_employees_enc_500-1000', 'no_employees_enc_6-25', 'no_employees_enc_More than 1000']
Nominal one-hot → remote_work expanded to ['remote_work_enc_Always', 'remote_work_enc_Never', 'remote_work_enc_No', 'remote_work_enc_Sometimes', 'remote_work_enc_Yes']
Ternary → benefits mapped to benefits_enc (0/1/2)
Ternary → care_options mapped to care_options_e

In [57]:
# Check problematic columns
df_clean.head()

,age,gender,country,self_employed,family_history,treatment,work_interfere,no_employees,remote_work,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_vs_physical,obs_consequence,comments,survey_year,age_group,self_employed_enc,treatment_enc,tech_company_enc,obs_consequence_enc,gender_enc_Female,gender_enc_Male,gender_enc_Other,family_history_enc,work_interfere_enc,no_employees_enc_1-5,no_employees_enc_100-500,no_employees_enc_26-100,no_employees_enc_500-1000,no_employees_enc_6-25,no_employees_enc_More than 1000,remote_work_enc_Always,remote_work_enc_Never,remote_work_enc_No,remote_work_enc_Sometimes,remote_work_enc_Yes,benefits_enc,care_options_enc,wellness_program_enc,seek_help_enc,anonymity_enc,leave_enc,mental_health_consequence_enc_Maybe,mental_health_consequence_enc_No,mental_health_consequence_enc_Yes,phys_health_consequence_enc_Maybe,phys_health_consequence_enc_No,phys_health_consequence_enc_Yes,coworkers_enc_No,coworkers_enc_Some of them,coworkers_enc_Yes,supervisor_enc_No,supervisor_enc_Some of them,supervisor_enc_Yes,mental_vs_physical_enc,age_group_enc_18-24,age_group_enc_25-34,age_group_enc_35-44,age_group_enc_45-54,age_group_enc_55-64,age_group_enc_65+
0,37.0,Female,United States,NaN,No,1,Often,6-25,No,1.0,Yes,Not sure,No,Yes,Yes,Somewhat easy,No,No,Some of them,Yes,Yes,0.0,NaN,2014,35-44,NaN,1,1.0,0.0,True,False,False,0,3.0,False,False,False,False,True,False,False,False,True,False,False,1,2,0,1,1,1.0,False,True,False,False,True,False,False,True,False,False,False,True,1,False,False,True,False,False,False
1,44.0,Male,United States,NaN,No,0,Rarely,More than 1000,No,0.0,Don't know,No,Don't know,Don't know,Don't know,Don't know,Maybe,No,No,No,Don't know,0.0,NaN,2014,35-44,NaN,0,0.0,0.0,False,True,False,0,1.0,False,False,False,False,False,True,False,False,True,False,False,2,0,2,2,2,NaN,True,False,False,False,True,False,True,False,False,True,False,False,2,False,False,True,False,False,False
2,32.0,Male,Canada,NaN,No,0,Rarely,6-25,No,1.0,No,No,No,No,Don't know,Somewhat difficult,No,No,Yes,Yes,No,0.0,NaN,2014,25-34,NaN,0,1.0,0.0,False,True,False,0,1.0,False,False,False,False,True,False,False,False,True,False,False,0,0,0,0,2,3.0,False,True,False,False,True,False,False,False,True,False,False,True,0,False,True,False,False,False,False
3,31.0,Male,United Kingdom,NaN,Yes,1,Often,26-100,No,1.0,No,Yes,No,No,No,Somewhat difficult,Yes,Yes,Some of them,No,No,1.0,NaN,2014,25-34,NaN,1,1.0,1.0,False,True,False,1,3.0,False,False,True,False,False,False,False,False,True,False,False,0,1,0,0,0,3.0,False,False,True,False,False,True,False,True,False,True,False,False,0,False,True,False,False,False,False
4,31.0,Male,United States,NaN,No,0,Never,100-500,Yes,1.0,Yes,No,Don't know,Don't know,Don't know,Don't know,No,No,Some of them,Yes,Don't know,0.0,NaN,2014,25-34,NaN,0,1.0,0.0,False,True,False,0,0.0,False,True,False,False,False,False,False,False,False,False,True,1,0,2,2,2,NaN,False,True,False,False,True,False,False,True,False,False,False,True,2,False,True,False,False,False,False


In [58]:
df_sorted = df_clean.sort_index(axis=1)
df_sorted.head()

,age,age_group,age_group_enc_18-24,age_group_enc_25-34,age_group_enc_35-44,age_group_enc_45-54,age_group_enc_55-64,age_group_enc_65+,anonymity,anonymity_enc,benefits,benefits_enc,care_options,care_options_enc,comments,country,coworkers,coworkers_enc_No,coworkers_enc_Some of them,coworkers_enc_Yes,family_history,family_history_enc,gender,gender_enc_Female,gender_enc_Male,gender_enc_Other,leave,leave_enc,mental_health_consequence,mental_health_consequence_enc_Maybe,mental_health_consequence_enc_No,mental_health_consequence_enc_Yes,mental_vs_physical,mental_vs_physical_enc,no_employees,no_employees_enc_1-5,no_employees_enc_100-500,no_employees_enc_26-100,no_employees_enc_500-1000,no_employees_enc_6-25,no_employees_enc_More than 1000,obs_consequence,obs_consequence_enc,phys_health_consequence,phys_health_consequence_enc_Maybe,phys_health_consequence_enc_No,phys_health_consequence_enc_Yes,remote_work,remote_work_enc_Always,remote_work_enc_Never,remote_work_enc_No,remote_work_enc_Sometimes,remote_work_enc_Yes,seek_help,seek_help_enc,self_employed,self_employed_enc,supervisor,supervisor_enc_No,supervisor_enc_Some of them,supervisor_enc_Yes,survey_year,tech_company,tech_company_enc,treatment,treatment_enc,wellness_program,wellness_program_enc,work_interfere,work_interfere_enc
0,37.0,35-44,False,False,True,False,False,False,Yes,1,Yes,1,Not sure,2,NaN,United States,Some of them,False,True,False,No,0,Female,True,False,False,Somewhat easy,1.0,No,False,True,False,Yes,1,6-25,False,False,False,False,True,False,0.0,0.0,No,False,True,False,No,False,False,True,False,False,Yes,1,NaN,NaN,Yes,False,False,True,2014,1.0,1.0,1,1,No,0,Often,3.0
1,44.0,35-44,False,False,True,False,False,False,Don't know,2,Don't know,2,No,0,NaN,United States,No,True,False,False,No,0,Male,False,True,False,Don't know,NaN,Maybe,True,False,False,Don't know,2,More than 1000,False,False,False,False,False,True,0.0,0.0,No,False,True,False,No,False,False,True,False,False,Don't know,2,NaN,NaN,No,True,False,False,2014,0.0,0.0,0,0,Don't know,2,Rarely,1.0
2,32.0,25-34,False,True,False,False,False,False,Don't know,2,No,0,No,0,NaN,Canada,Yes,False,False,True,No,0,Male,False,True,False,Somewhat difficult,3.0,No,False,True,False,No,0,6-25,False,False,False,False,True,False,0.0,0.0,No,False,True,False,No,False,False,True,False,False,No,0,NaN,NaN,Yes,False,False,True,2014,1.0,1.0,0,0,No,0,Rarely,1.0
3,31.0,25-34,False,True,False,False,False,False,No,0,No,0,Yes,1,NaN,United Kingdom,Some of them,False,True,False,Yes,1,Male,False,True,False,Somewhat difficult,3.0,Yes,False,False,True,No,0,26-100,False,False,True,False,False,False,1.0,1.0,Yes,False,False,True,No,False,False,True,False,False,No,0,NaN,NaN,No,True,False,False,2014,1.0,1.0,1,1,No,0,Often,3.0
4,31.0,25-34,False,True,False,False,False,False,Don't know,2,Yes,1,No,0,NaN,United States,Some of them,False,True,False,No,0,Male,False,True,False,Don't know,NaN,No,False,True,False,Don't know,2,100-500,False,True,False,False,False,False,0.0,0.0,No,False,True,False,Yes,False,False,False,False,True,Don't know,2,NaN,NaN,Yes,False,False,True,2014,1.0,1.0,0,0,Don't know,2,Never,0.0


##### Caclulating support_score

In [59]:
df_clean['support_score'] = (df_clean['benefits_enc'] == 1).astype(int) + \
                            (df_clean['care_options_enc'] == 1).astype(int) + \
                            (df_clean['wellness_program_enc'] == 1).astype(int)

In [60]:
df_clean.groupby('support_score').size()

support_score
0    2376
1    1039
2     811
3     451
dtype: int64

##### benefits, care options, and wellness programs are categorical, with (==2) meaning 'I don't know'
##### To avoid inflation and misleading results, support_score will only sum explicit yes's (==1)

In [61]:
df_clean.isna().sum().sort_values(ascending=False).reset_index()

,index,0
0,comments,4516
1,phys_health_consequence,3426
2,mental_health_consequence,3426
3,obs_consequence_enc,3426
4,obs_consequence,3426
5,supervisor,3426
6,coworkers,3426
7,work_interfere_enc,2726
8,remote_work,1998
9,mental_vs_physical,1723


In [62]:
for col in df_clean.select_dtypes(include=['object', 'category','float64', 'int64']).columns:
    print(f"--- {col} ---")
    print(df_clean[col].unique())
    print()
    print("------------------------------------------------------------------------------")

--- age ---
[37. 44. 32. 31. 33. 35. 39. 42. 23. 29. 36. 27. 46. 41. 34. 30. 40. 38.
 50. 24. 18. 28. 26. 22. 19. 25. 45. 21. 43. 56. 60. 54. 55. 48. 20. 57.
 58. 47. 62. 51. 65. 49. 53. 61. 72. 52. 63. 66. 59. 74. 70. 64. 67. 76.]

------------------------------------------------------------------------------
--- gender ---
['Female' 'Male' 'Other']

------------------------------------------------------------------------------
--- country ---
['United States' 'Canada' 'United Kingdom' 'Bulgaria' 'France' 'Portugal'
 'Netherlands' 'Switzerland' 'Poland' 'Australia' 'Germany' 'Russia'
 'Mexico' 'Brazil' 'Slovenia' 'Costa Rica' 'Austria' 'Ireland' 'India'
 'South Africa' 'Italy' 'Sweden' 'Colombia' 'Latvia' 'Romania' 'Belgium'
 'New Zealand' 'Spain' 'Finland' 'Uruguay' 'Israel'
 'Bosnia and Herzegovina' 'Hungary' 'Singapore' 'Japan' 'Nigeria'
 'Croatia' 'Norway' 'Thailand' 'Denmark' 'Greece' 'Moldova' 'Georgia'
 'China' 'Czech Republic' 'Philippines' 'United States of America'
 'Lithuan

### Save data for EDA

In [63]:
df_clean.to_csv(PROCESSED_DATA / 'cleaned_data.csv', index=False)